# Project 01 — Exploratory Data Analysis: Netflix Movies & TV Shows
**Pluto Academy AI & ML Internship**

**Dataset:** [Netflix Movies and TV Shows — Kaggle](https://www.kaggle.com/datasets/shivamb/netflix-shows)

**Objective:** Perform a complete EDA on the Netflix dataset — clean the data, answer 5 analytical questions, create 6+ visualizations, and derive 5 business insights.

---

## Step 0 — Install & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries imported successfully.')

## Step 1 — Load & Inspect the Data

> **How to get the dataset:**
> 1. Go to https://www.kaggle.com/datasets/shivamb/netflix-shows
> 2. Download `netflix_titles.csv`
> 3. Upload it to this Colab session using the file upload button on the left sidebar, OR mount Google Drive and place it there.

In [ ]:
# --- Option A: Upload directly to Colab session ---
# from google.colab import files
# uploaded = files.upload()  # uncomment and run to upload manually

# --- Option B: Load from Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/netflix_titles.csv')

# --- Default: Load from current session directory ---
df = pd.read_csv('netflix_titles.csv')

print('Dataset loaded successfully.')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')

In [ ]:
# Preview first 5 rows
df.head()

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Missing value summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False))

In [ ]:
# Duplicate check
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Unique show IDs: {df["show_id"].nunique()}')

### Data Summary (5-line overview)

1. The dataset contains **8,807 titles** across **12 columns**, covering Netflix content from multiple countries.
2. Each row represents one title with attributes: type (Movie/TV Show), title, director, cast, country, date added, release year, rating, duration, genres (listed_in), and description.
3. The columns with the most missing values are `director` (~30%), `cast` (~10%), and `country` (~7%) — these are expected gaps for older or international content.
4. There are **no duplicate rows**, and `show_id` is fully unique.
5. The content spans release years from **1925 to 2021**, with most content added between 2015–2021.

---
## Step 2 — Clean the Data

In [ ]:
# --- Cleaning Decision 1: Fill missing director/cast/country with 'Unknown' ---
# Reason: These are categorical fields. Dropping rows would lose ~30% of data unnecessarily.
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')

# --- Cleaning Decision 2: Drop rows where 'date_added' or 'rating' is missing ---
# Reason: Only ~11 rows each; small enough to drop without affecting analysis.
df.dropna(subset=['date_added', 'rating'], inplace=True)

# --- Cleaning Decision 3: Parse date_added to datetime ---
# Reason: Enables year/month-based time series analysis.
df['date_added'] = pd.to_datetime(df['date_added'].str.strip())
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

# --- Cleaning Decision 4: Extract numeric duration ---
# Reason: 'duration' is a string like '90 min' or '2 Seasons'. Split into numeric + unit.
df['duration_value'] = df['duration'].str.extract(r'(\d+)').astype(float)
df['duration_unit'] = df['duration'].str.extract(r'([A-Za-z]+)')

# --- Cleaning Decision 5: Drop irrelevant column 'show_id' ---
# Reason: It's a row identifier with no analytical value.
df.drop(columns=['show_id'], inplace=True)

print('Cleaning complete.')
print(f'Shape after cleaning: {df.shape}')
print(f'Remaining nulls:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

---
## Step 3 — Exploratory Data Analysis (5 Questions)

### Q1: What is the split between Movies and TV Shows on Netflix?

In [ ]:
type_counts = df['type'].value_counts()
print('Content Type Distribution:')
print(type_counts)
print(f'\nMovies account for {type_counts["Movie"]/len(df)*100:.1f}% of all content.')

### Q2: Which are the top 10 countries producing Netflix content?

In [ ]:
# Some entries list multiple countries; take the first (primary) country
df['primary_country'] = df['country'].str.split(',').str[0].str.strip()
top_countries = df[df['primary_country'] != 'Unknown']['primary_country'].value_counts().head(10)
print('Top 10 Content-Producing Countries:')
print(top_countries)

### Q3: How has Netflix's content addition grown year-over-year?

In [ ]:
yearly_additions = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)
print('Titles Added Per Year by Type:')
print(yearly_additions)

### Q4: What are the most common content ratings on Netflix?

In [ ]:
rating_counts = df['rating'].value_counts()
print('Content Rating Distribution:')
print(rating_counts)

### Q5: What are the top 10 most common genres on Netflix?

In [ ]:
# Each title can have multiple genres; explode to count individually
genres = df['listed_in'].str.split(', ').explode()
top_genres = genres.value_counts().head(10)
print('Top 10 Genres:')
print(top_genres)

---
## Step 4 — Visualizations (6 Charts)

### Chart 1 — Pie Chart: Movie vs TV Show Split

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
colors = ['#E50914', '#221F1F']
explode = (0.05, 0)
ax.pie(
    type_counts,
    labels=type_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    explode=explode,
    startangle=90,
    textprops={'fontsize': 13}
)
ax.set_title('Netflix Content: Movie vs TV Show Split', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('chart1_pie_type_split.png', bbox_inches='tight')
plt.show()

### Chart 2 — Bar Chart: Top 10 Content-Producing Countries

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
top_countries.sort_values().plot(kind='barh', ax=ax, color='#E50914', edgecolor='black')
ax.set_title('Top 10 Countries by Netflix Content Volume', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Titles', fontsize=12)
ax.set_ylabel('Country', fontsize=12)
for i, v in enumerate(top_countries.sort_values()):
    ax.text(v + 10, i, str(v), va='center', fontsize=10)
plt.tight_layout()
plt.savefig('chart2_bar_top_countries.png', bbox_inches='tight')
plt.show()

### Chart 3 — Line Chart: Content Addition Growth Over Years

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
yearly_plot = yearly_additions[yearly_additions.index >= 2010]
for col, color in zip(yearly_plot.columns, ['#E50914', '#3498db']):
    ax.plot(yearly_plot.index, yearly_plot[col], marker='o', label=col, color=color, linewidth=2.5)
ax.set_title('Netflix Content Added Per Year (2010–2021)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Number of Titles Added', fontsize=12)
ax.legend(title='Type', fontsize=11)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.tight_layout()
plt.savefig('chart3_line_yearly_growth.png', bbox_inches='tight')
plt.show()

### Chart 4 — Bar Chart: Content Rating Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
rating_counts.plot(kind='bar', ax=ax, color=sns.color_palette('Reds_r', len(rating_counts)), edgecolor='black')
ax.set_title('Distribution of Content Ratings on Netflix', fontsize=14, fontweight='bold')
ax.set_xlabel('Rating', fontsize=12)
ax.set_ylabel('Number of Titles', fontsize=12)
ax.tick_params(axis='x', rotation=45)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x() + p.get_width()/2, p.get_height() + 10),
                ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('chart4_bar_ratings.png', bbox_inches='tight')
plt.show()

### Chart 5 — Histogram: Movie Duration Distribution

In [ ]:
movies_df = df[(df['type'] == 'Movie') & (df['duration_value'].notna())]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(movies_df['duration_value'], bins=40, color='#E50914', edgecolor='black', alpha=0.85)
ax.axvline(movies_df['duration_value'].median(), color='navy', linestyle='--', linewidth=2,
           label=f'Median: {movies_df["duration_value"].median():.0f} min')
ax.set_title('Distribution of Netflix Movie Durations', fontsize=14, fontweight='bold')
ax.set_xlabel('Duration (minutes)', fontsize=12)
ax.set_ylabel('Number of Movies', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('chart5_histogram_movie_duration.png', bbox_inches='tight')
plt.show()

### Chart 6 — Horizontal Bar Chart: Top 10 Genres

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top_genres.sort_values().plot(kind='barh', ax=ax,
                               color=sns.color_palette('YlOrRd', len(top_genres)),
                               edgecolor='black')
ax.set_title('Top 10 Genres on Netflix', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Titles', fontsize=12)
ax.set_ylabel('Genre', fontsize=12)
for i, v in enumerate(top_genres.sort_values()):
    ax.text(v + 15, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('chart6_bar_top_genres.png', bbox_inches='tight')
plt.show()

### Chart 7 (Bonus) — Heatmap: Monthly Content Additions by Year

In [ ]:
heatmap_data = df[df['year_added'] >= 2015].groupby(['year_added', 'month_added']).size().unstack(fill_value=0)
heatmap_data.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='Reds', ax=ax, linewidths=0.4)
ax.set_title('Netflix Titles Added per Month (2015–2021)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Year', fontsize=12)
plt.tight_layout()
plt.savefig('chart7_heatmap_monthly.png', bbox_inches='tight')
plt.show()

---
## Step 5 — Insights Report

### 5 Business Insights from the Netflix EDA

**Insight 1: Netflix is primarily a movie platform, but TV Shows are growing.**
As shown in Chart 1 (pie chart), ~69% of Netflix content is movies. However, Chart 3 (line chart) shows that TV Show additions have grown steadily since 2016 — signaling a strategic shift toward serialized content to improve subscriber retention.

**Insight 2: The United States dominates Netflix's content library by a massive margin.**
Chart 2 (bar chart) shows the US contributes more titles than the next 4 countries combined. India ranks 2nd, reflecting Netflix's major investment in Bollywood and regional Indian content — a high-growth market.

**Insight 3: Netflix's content addition accelerated sharply between 2016 and 2019, then slowed during COVID.**
Chart 3 (line chart) shows content additions peaked around 2019. A visible dip in 2020 aligns with pandemic-era production shutdowns, suggesting Netflix's pipeline is heavily dependent on live production capacity.

**Insight 4: Most Netflix content targets mature audiences — TV-MA and TV-14 dominate.**
Chart 4 (bar chart) reveals TV-MA is the single most common rating, comprising over 35% of content. This indicates Netflix's primary audience is adults 18+, and family/children's programming is underrepresented relative to total catalog size.

**Insight 5: The typical Netflix movie runs between 85–100 minutes — tightly clustered around the 90-minute sweet spot.**
Chart 5 (histogram) shows a clear normal distribution peaking near 90 minutes (median line). This suggests Netflix commissioning follows traditional cinema runtime norms — future analysis could compare original vs licensed content durations.

---

### Most Surprising Finding

The most surprising finding was the **sharp dip in content additions in 2020** (visible in Chart 3 and the heatmap). Despite COVID-19 driving a massive surge in streaming subscriptions and viewership, Netflix actually added *fewer* titles that year compared to 2019 — likely because global production was halted. This reveals a clear tension between demand spikes and supply constraints in the streaming industry.

---
## Summary Statistics

In [ ]:
print('=== FINAL DATASET SUMMARY ===')
print(f'Total titles after cleaning: {len(df)}')
print(f'Movies: {(df["type"]=="Movie").sum()} | TV Shows: {(df["type"]=="TV Show").sum()}')
print(f'Date range of additions: {df["date_added"].min().date()} to {df["date_added"].max().date()}')
print(f'Unique countries: {df["primary_country"].nunique()}')
print(f'Unique genres: {genres.nunique()}')
print(f'Average movie duration: {movies_df["duration_value"].mean():.1f} minutes')